# LangChain 综合实战与优化学习Demo

本Notebook将综合运用前面学到的所有知识，构建真实的AI应用案例。

## 学习大纲
1. **RAG + Agent集成** - 构建知识库问答Agent
2. **Multi-Agent + LangGraph** - 复杂的多Agent协作系统
3. **性能优化技术** - 缓存、批处理、异步
4. **生产级实践** - 错误处理、监控、评估
5. **完整项目案例** - 智能文档分析助手

## 环境准备

In [ ]:
# 安装依赖
# pip install langchain==0.3.15
# pip install langchain-core==0.3.28
# pip install langchain-community==0.3.14
# pip install langgraph==0.2.64
# pip install dashscope==1.20.11
# pip install chromadb==0.5.23
# pip install tiktoken==0.8.0

## 导入库并配置API Key

In [ ]:
import os
from dotenv import load_dotenv
import time
from typing import TypedDict, List, Annotated

# 加载环境变量
load_dotenv()
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# 验证API Key
if DASHSCOPE_API_KEY:
    print("✅ API Key已加载")
else:
    print("❌ 请在.env文件中配置DASHSCOPE_API_KEY")

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi
from langchain_community.embeddings import DashScopeEmbeddings

# 初始化LLM和Embeddings
llm = ChatTongyi(
    model="qwen-plus",
    temperature=0.7,
    dashscope_api_key=DASHSCOPE_API_KEY
)

embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=DASHSCOPE_API_KEY
)

print("✅ LLM和Embeddings初始化完成")

---

# 第一部分：RAG + Agent 集成

构建一个能够自主检索、分析和回答的智能Agent。

## 1.1 构建知识库

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 创建示例知识库文档
knowledge_docs = [
    Document(
        page_content="""LangChain是一个强大的框架，用于开发由大型语言模型(LLM)驱动的应用程序。
它提供了模块化的组件，包括模型I/O、数据连接、链(Chains)、代理(Agents)、内存等。
LangChain支持多种LLM提供商，如OpenAI、Anthropic、阿里云通义千问等。""",
        metadata={"source": "langchain_intro", "category": "framework"}
    ),
    Document(
        page_content="""RAG(检索增强生成)是一种结合信息检索和生成的技术。
它通过先检索相关文档，然后将检索到的上下文与用户问题一起输入LLM来生成答案。
RAG的优势包括：减少幻觉、提供可追溯的信息来源、支持领域知识扩展等。""",
        metadata={"source": "rag_guide", "category": "technique"}
    ),
    Document(
        page_content="""Agent是能够自主决策和行动的AI系统。
它可以使用工具、观察环境、进行推理并执行动作。
常见的Agent模式包括ReAct、Plan-and-Execute、Self-Ask等。
LangChain提供了完整的Agent开发框架。""",
        metadata={"source": "agent_guide", "category": "technique"}
    ),
    Document(
        page_content="""LangGraph是LangChain的状态图框架，用于构建复杂的、有状态的AI应用。
它支持循环、条件分支、人机协作等高级功能。
LangGraph特别适合构建需要多步推理、自我反思或人工审核的应用。""",
        metadata={"source": "langgraph_guide", "category": "framework"}
    ),
    Document(
        page_content="""向量数据库用于存储和检索高维向量。
常见的向量数据库包括Chroma、FAISS、Pinecone、Weaviate等。
它们支持快速的相似度搜索，是RAG系统的核心组件。
选择向量数据库时需要考虑性能、可扩展性、成本等因素。""",
        metadata={"source": "vector_db_guide", "category": "infrastructure"}
    ),
    Document(
        page_content="""提示工程(Prompt Engineering)是优化LLM输出的关键技术。
有效的提示应该清晰、具体，并提供必要的上下文。
常用技巧包括：few-shot示例、思维链(Chain-of-Thought)、角色设定等。
LangChain的PromptTemplate可以帮助标准化和管理提示。""",
        metadata={"source": "prompt_engineering", "category": "technique"}
    )
]

print(f"✅ 创建了 {len(knowledge_docs)} 个知识文档")

In [ ]:
# 分割文档
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "，", " ", ""]
)

splits = text_splitter.split_documents(knowledge_docs)
print(f"✅ 文档分割完成: {len(splits)} 个chunks")

# 创建向量库
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="rag_agent_kb"
)

print("✅ 向量库创建完成")

## 1.2 创建RAG工具

In [ ]:
from langchain_core.tools import tool
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import BM25Retriever

@tool
def search_knowledge_base(query: str) -> str:
    """在知识库中搜索相关信息
    
    Args:
        query: 搜索查询字符串
    
    Returns:
        检索到的相关信息
    """
    print(f"🔍 [知识库搜索] 查询: {query}")
    
    # 向量检索
    vector_retriever = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 6}
    )
    
    # BM25检索
    bm25_retriever = BM25Retriever.from_documents(splits)
    bm25_retriever.k = 3
    
    # 混合检索
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[0.4, 0.6]
    )
    
    docs = ensemble_retriever.invoke(query)
    
    # 格式化结果
    result = "检索到的相关信息:\n\n"
    for i, doc in enumerate(docs[:3], 1):
        result += f"[{i}] {doc.page_content}\n"
        result += f"    来源: {doc.metadata.get('source', 'unknown')}\n\n"
    
    print(f"✅ 检索到 {len(docs)} 个相关文档")
    return result

@tool
def analyze_document_category(query: str) -> str:
    """分析查询相关的文档类别
    
    Args:
        query: 查询字符串
    
    Returns:
        文档类别统计
    """
    print(f"📊 [类别分析] 分析查询: {query}")
    
    docs = vectorstore.similarity_search(query, k=5)
    
    categories = {}
    for doc in docs:
        cat = doc.metadata.get('category', 'unknown')
        categories[cat] = categories.get(cat, 0) + 1
    
    result = "文档类别分布:\n"
    for cat, count in categories.items():
        result += f"- {cat}: {count}个文档\n"
    
    return result

print("✅ RAG工具创建完成")

## 1.3 创建RAG Agent

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# 定义工具列表
rag_tools = [search_knowledge_base, analyze_document_category]

# 创建Agent提示词
rag_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """你是一个专业的知识库助手。你可以使用以下工具来回答用户问题:

{tools}

工作流程:
1. 先使用search_knowledge_base工具检索相关信息
2. 必要时使用analyze_document_category了解信息分布
3. 基于检索到的信息提供准确、详细的回答
4. 如果知识库中没有相关信息，请明确说明

请始终基于事实回答，不要编造信息。"""),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 创建Agent
rag_agent = create_tool_calling_agent(llm, rag_tools, rag_agent_prompt)

# 创建Executor
rag_agent_executor = AgentExecutor(
    agent=rag_agent,
    tools=rag_tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,
)

print("✅ RAG Agent创建完成")

In [ ]:
# 测试RAG Agent
test_questions = [
    "什么是RAG？它有什么优势？",
    "LangGraph和LangChain有什么关系？",
    "如何选择合适的向量数据库？"
]

for question in test_questions:
    print("\n" + "="*60)
    print(f"❓ 问题: {question}")
    print("="*60)
    
    result = rag_agent_executor.invoke({"input": question})
    
    print(f"\n💡 回答: {result['output']}\n")

---

# 第二部分：Multi-Agent + LangGraph 集成

构建一个复杂的文档分析系统，多个Agent协作完成任务。

## 2.1 定义状态和Agent

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import Literal

# 定义系统状态
class DocumentAnalysisState(TypedDict):
    """文档分析系统状态"""
    document_text: str  # 原始文档
    query: str  # 用户查询
    
    # 检索阶段
    retrieved_info: str
    
    # 分析阶段
    analysis_type: str  # "summary", "qa", "extract"
    analysis_result: str
    
    # 验证阶段
    quality_score: float
    needs_revision: bool
    
    # 最终结果
    final_answer: str
    iteration: int

print("✅ 状态定义完成")

In [ ]:
# 定义各个Agent节点

def retrieval_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """检索Agent - 从知识库检索相关信息"""
    print(f"\n🔍 [检索Agent] 检索相关信息...")
    
    # 使用知识库检索
    docs = vectorstore.similarity_search(state["query"], k=3)
    
    retrieved_info = "检索到的信息:\n"
    for i, doc in enumerate(docs, 1):
        retrieved_info += f"\n[{i}] {doc.page_content}\n"
    
    state["retrieved_info"] = retrieved_info
    print(f"✅ 检索完成，获得 {len(docs)} 个文档")
    return state

def classification_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """分类Agent - 判断查询类型"""
    print(f"\n🏷️  [分类Agent] 分析查询类型...")
    
    query = state["query"].lower()
    
    if any(word in query for word in ["总结", "摘要", "概括"]):
        state["analysis_type"] = "summary"
        print("✅ 类型: 摘要")
    elif any(word in query for word in ["提取", "找出", "列出"]):
        state["analysis_type"] = "extract"
        print("✅ 类型: 信息提取")
    else:
        state["analysis_type"] = "qa"
        print("✅ 类型: 问答")
    
    return state

def summary_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """摘要Agent - 生成摘要"""
    print(f"\n📝 [摘要Agent] 生成摘要...")
    
    prompt = f"""基于以下信息生成简洁的摘要:

{state['retrieved_info']}

用户问题: {state['query']}

请提供清晰、简洁的摘要。"""
    
    response = llm.invoke(prompt)
    state["analysis_result"] = response.content
    print("✅ 摘要生成完成")
    return state

def extraction_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """提取Agent - 提取关键信息"""
    print(f"\n🎯 [提取Agent] 提取关键信息...")
    
    prompt = f"""从以下信息中提取关键要点:

{state['retrieved_info']}

用户需求: {state['query']}

请以列表形式列出关键信息。"""
    
    response = llm.invoke(prompt)
    state["analysis_result"] = response.content
    print("✅ 信息提取完成")
    return state

def qa_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """问答Agent - 回答问题"""
    print(f"\n💬 [问答Agent] 生成答案...")
    
    prompt = f"""基于以下上下文回答用户问题:

上下文:
{state['retrieved_info']}

问题: {state['query']}

请提供准确、详细的答案。"""
    
    response = llm.invoke(prompt)
    state["analysis_result"] = response.content
    print("✅ 答案生成完成")
    return state

def quality_check_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """质量检查Agent - 评估回答质量"""
    print(f"\n✅ [质检Agent] 评估回答质量...")
    
    prompt = f"""评估以下回答的质量（1-10分）:

问题: {state['query']}
回答: {state['analysis_result']}

评估标准:
- 准确性
- 完整性
- 清晰度

只输出分数（1-10的整数）。"""
    
    try:
        response = llm.invoke(prompt)
        # 尝试提取数字
        score_text = response.content.strip()
        score = float(''.join(filter(str.isdigit, score_text)))
        score = min(10, max(1, score))  # 确保在1-10范围内
    except:
        score = 7.0  # 默认分数
    
    state["quality_score"] = score
    state["needs_revision"] = score < 7.0
    
    print(f"评分: {score}/10, 需要修订: {state['needs_revision']}")
    return state

def finalize_agent(state: DocumentAnalysisState) -> DocumentAnalysisState:
    """最终化Agent - 生成最终答案"""
    print(f"\n📋 [最终化Agent] 生成最终答案...")
    
    state["final_answer"] = f"""问题: {state['query']}

回答:
{state['analysis_result']}

质量评分: {state['quality_score']}/10
迭代次数: {state['iteration']}
"""
    print("✅ 最终答案生成完成")
    return state

print("✅ Agent节点定义完成")

## 2.2 构建LangGraph工作流

In [ ]:
# 定义路由函数
def route_by_analysis_type(state: DocumentAnalysisState) -> str:
    """根据分析类型路由"""
    return state["analysis_type"]

def check_quality(state: DocumentAnalysisState) -> Literal["finalize", "retry"]:
    """检查质量是否合格"""
    max_iterations = 2
    
    if state["needs_revision"] and state["iteration"] < max_iterations:
        state["iteration"] += 1
        return "retry"
    return "finalize"

# 构建工作流
workflow = StateGraph(DocumentAnalysisState)

# 添加所有节点
workflow.add_node("retrieve", retrieval_agent)
workflow.add_node("classify", classification_agent)
workflow.add_node("summary", summary_agent)
workflow.add_node("extract", extraction_agent)
workflow.add_node("qa", qa_agent)
workflow.add_node("quality_check", quality_check_agent)
workflow.add_node("finalize", finalize_agent)

# 设置入口
workflow.set_entry_point("retrieve")

# 添加边
workflow.add_edge("retrieve", "classify")

# 条件路由到不同的分析Agent
workflow.add_conditional_edges(
    "classify",
    route_by_analysis_type,
    {
        "summary": "summary",
        "extract": "extract",
        "qa": "qa"
    }
)

# 所有分析Agent都连接到质检
workflow.add_edge("summary", "quality_check")
workflow.add_edge("extract", "quality_check")
workflow.add_edge("qa", "quality_check")

# 质检后决定是否重试
workflow.add_conditional_edges(
    "quality_check",
    check_quality,
    {
        "finalize": "finalize",
        "retry": "classify"  # 重新分类和分析
    }
)

workflow.add_edge("finalize", END)

# 编译
doc_analysis_app = workflow.compile()

print("✅ 文档分析工作流构建完成")

In [ ]:
# 测试文档分析系统
test_cases = [
    "总结一下LangChain的主要功能",
    "RAG有哪些优势？",
    "提取关于Agent的关键信息"
]

for query in test_cases:
    print("\n" + "="*70)
    print(f"🎯 查询: {query}")
    print("="*70)
    
    result = doc_analysis_app.invoke({
        "document_text": "",
        "query": query,
        "retrieved_info": "",
        "analysis_type": "",
        "analysis_result": "",
        "quality_score": 0.0,
        "needs_revision": False,
        "final_answer": "",
        "iteration": 0
    })
    
    print("\n" + "="*70)
    print("📄 最终结果")
    print("="*70)
    print(result["final_answer"])
    print()

---

# 第三部分：性能优化技术

## 3.1 缓存优化

In [ ]:
from functools import lru_cache
import hashlib

class CachedRetriever:
    """带缓存的检索器"""
    
    def __init__(self, vectorstore, cache_size=128):
        self.vectorstore = vectorstore
        self.cache = {}
        self.cache_size = cache_size
        self.hits = 0
        self.misses = 0
    
    def _get_cache_key(self, query: str) -> str:
        """生成缓存键"""
        return hashlib.md5(query.encode()).hexdigest()
    
    def retrieve(self, query: str, k: int = 3) -> List:
        """带缓存的检索"""
        cache_key = self._get_cache_key(query)
        
        # 检查缓存
        if cache_key in self.cache:
            self.hits += 1
            print(f"💾 缓存命中! (命中率: {self.hit_rate():.1%})")
            return self.cache[cache_key]
        
        # 缓存未命中，执行检索
        self.misses += 1
        docs = self.vectorstore.similarity_search(query, k=k)
        
        # 存入缓存
        if len(self.cache) >= self.cache_size:
            # 简单的FIFO策略，删除最早的条目
            self.cache.pop(next(iter(self.cache)))
        
        self.cache[cache_key] = docs
        print(f"🔍 缓存未命中，执行检索 (命中率: {self.hit_rate():.1%})")
        
        return docs
    
    def hit_rate(self) -> float:
        """计算缓存命中率"""
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0
    
    def stats(self) -> dict:
        """获取缓存统计"""
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": self.hit_rate(),
            "cache_size": len(self.cache)
        }

# 创建缓存检索器
cached_retriever = CachedRetriever(vectorstore)

print("✅ 缓存检索器创建完成")

In [ ]:
# 测试缓存效果
print("\n" + "="*60)
print("测试缓存性能")
print("="*60 + "\n")

queries = [
    "什么是RAG?",
    "LangChain是什么?",
    "什么是RAG?",  # 重复查询
    "Agent的作用",
    "LangChain是什么?",  # 重复查询
]

for query in queries:
    print(f"\n查询: {query}")
    start_time = time.time()
    docs = cached_retriever.retrieve(query)
    elapsed = time.time() - start_time
    print(f"耗时: {elapsed*1000:.2f}ms, 结果数: {len(docs)}")

print("\n" + "="*60)
print("缓存统计")
print("="*60)
stats = cached_retriever.stats()
print(f"命中次数: {stats['hits']}")
print(f"未命中次数: {stats['misses']}")
print(f"命中率: {stats['hit_rate']:.1%}")
print(f"缓存大小: {stats['cache_size']}")

## 3.2 批处理优化

In [ ]:
def batch_embed_documents(texts: List[str], batch_size: int = 10) -> List:
    """批量embedding处理"""
    print(f"\n📦 批处理 {len(texts)} 个文档，批大小: {batch_size}")
    
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        print(f"   处理批次 {i//batch_size + 1}/{(len(texts)-1)//batch_size + 1}")
        
        # 批量embedding
        batch_embeddings = embeddings.embed_documents(batch)
        all_embeddings.extend(batch_embeddings)
        
        # 避免API限流
        time.sleep(0.1)
    
    print(f"✅ 批处理完成，生成 {len(all_embeddings)} 个embeddings")
    return all_embeddings

# 测试批处理
test_texts = [
    "这是测试文本1",
    "这是测试文本2",
    "这是测试文本3",
    "这是测试文本4",
    "这是测试文本5",
]

start_time = time.time()
embeddings_result = batch_embed_documents(test_texts, batch_size=2)
elapsed = time.time() - start_time

print(f"\n总耗时: {elapsed:.2f}秒")
print(f"平均每个文档: {elapsed/len(test_texts)*1000:.2f}ms")

---

# 第四部分：生产级实践

## 4.1 错误处理和重试机制

In [ ]:
from typing import Callable, Any
import traceback

class RobustAgent:
    """带错误处理和重试的健壮Agent"""
    
    def __init__(self, llm, max_retries: int = 3):
        self.llm = llm
        self.max_retries = max_retries
        self.error_count = 0
        self.success_count = 0
    
    def execute_with_retry(self, func: Callable, *args, **kwargs) -> Any:
        """带重试机制的执行"""
        last_error = None
        
        for attempt in range(self.max_retries):
            try:
                result = func(*args, **kwargs)
                self.success_count += 1
                return {"success": True, "result": result}
            
            except Exception as e:
                last_error = e
                self.error_count += 1
                
                print(f"⚠️  尝试 {attempt + 1}/{self.max_retries} 失败: {str(e)}")
                
                if attempt < self.max_retries - 1:
                    wait_time = 2 ** attempt  # 指数退避
                    print(f"   等待 {wait_time}秒后重试...")
                    time.sleep(wait_time)
        
        # 所有重试都失败
        return {
            "success": False,
            "error": str(last_error),
            "traceback": traceback.format_exc()
        }
    
    def safe_llm_call(self, prompt: str) -> dict:
        """安全的LLM调用"""
        def _call():
            response = self.llm.invoke(prompt)
            if not response or not response.content:
                raise ValueError("空响应")
            return response.content
        
        return self.execute_with_retry(_call)
    
    def get_stats(self) -> dict:
        """获取统计信息"""
        total = self.success_count + self.error_count
        return {
            "success_count": self.success_count,
            "error_count": self.error_count,
            "success_rate": self.success_count / total if total > 0 else 0
        }

# 创建健壮Agent
robust_agent = RobustAgent(llm)

print("✅ 健壮Agent创建完成")

In [ ]:
# 测试错误处理
print("\n" + "="*60)
print("测试错误处理和重试机制")
print("="*60 + "\n")

result = robust_agent.safe_llm_call("用一句话解释什么是LangChain")

if result["success"]:
    print(f"✅ 执行成功")
    print(f"结果: {result['result']}")
else:
    print(f"❌ 执行失败")
    print(f"错误: {result['error']}")

print("\n" + "="*60)
print("执行统计")
print("="*60)
stats = robust_agent.get_stats()
print(f"成功次数: {stats['success_count']}")
print(f"失败次数: {stats['error_count']}")
print(f"成功率: {stats['success_rate']:.1%}")

## 4.2 监控和日志

In [ ]:
import logging
from datetime import datetime

class AgentMonitor:
    """Agent监控器"""
    
    def __init__(self):
        self.metrics = {
            "total_requests": 0,
            "successful_requests": 0,
            "failed_requests": 0,
            "total_tokens": 0,
            "total_latency": 0.0,
            "requests_by_type": {}
        }
        
        # 配置日志
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger("AgentMonitor")
    
    def log_request(self, request_type: str, success: bool, latency: float, tokens: int = 0):
        """记录请求"""
        self.metrics["total_requests"] += 1
        
        if success:
            self.metrics["successful_requests"] += 1
        else:
            self.metrics["failed_requests"] += 1
        
        self.metrics["total_tokens"] += tokens
        self.metrics["total_latency"] += latency
        
        # 按类型统计
        if request_type not in self.metrics["requests_by_type"]:
            self.metrics["requests_by_type"][request_type] = 0
        self.metrics["requests_by_type"][request_type] += 1
        
        # 记录日志
        status = "SUCCESS" if success else "FAILED"
        self.logger.info(
            f"Request {status} - Type: {request_type}, "
            f"Latency: {latency*1000:.2f}ms, Tokens: {tokens}"
        )
    
    def get_metrics(self) -> dict:
        """获取指标"""
        total = self.metrics["total_requests"]
        
        return {
            **self.metrics,
            "success_rate": self.metrics["successful_requests"] / total if total > 0 else 0,
            "avg_latency": self.metrics["total_latency"] / total if total > 0 else 0,
            "avg_tokens": self.metrics["total_tokens"] / total if total > 0 else 0
        }
    
    def print_summary(self):
        """打印摘要"""
        metrics = self.get_metrics()
        
        print("\n" + "="*60)
        print("📊 监控摘要")
        print("="*60)
        print(f"总请求数: {metrics['total_requests']}")
        print(f"成功: {metrics['successful_requests']} ({metrics['success_rate']:.1%})")
        print(f"失败: {metrics['failed_requests']}")
        print(f"平均延迟: {metrics['avg_latency']*1000:.2f}ms")
        print(f"总Token数: {metrics['total_tokens']}")
        print(f"平均Token数: {metrics['avg_tokens']:.0f}")
        print("\n请求类型分布:")
        for req_type, count in metrics['requests_by_type'].items():
            print(f"  {req_type}: {count}")
        print("="*60)

# 创建监控器
monitor = AgentMonitor()

print("✅ 监控器创建完成")

In [ ]:
# 测试监控
print("\n测试监控功能\n")

# 模拟一些请求
for i in range(5):
    request_type = ["search", "qa", "summary"][i % 3]
    start_time = time.time()
    
    # 模拟请求
    result = llm.invoke(f"简短回答: 什么是{request_type}?")
    
    latency = time.time() - start_time
    tokens = len(result.content)  # 简化的token计数
    
    monitor.log_request(
        request_type=request_type,
        success=True,
        latency=latency,
        tokens=tokens
    )
    
    time.sleep(0.5)

# 打印监控摘要
monitor.print_summary()

---

# 总结

## 本课程学到的内容

### 1. RAG + Agent 集成
- ✅ 构建知识库并集成到Agent
- ✅ 实现混合检索策略
- ✅ 创建专业的知识库问答系统

### 2. Multi-Agent + LangGraph
- ✅ 设计复杂的多Agent协作系统
- ✅ 使用LangGraph编排工作流
- ✅ 实现质量检查和迭代改进

### 3. 性能优化
- ✅ 缓存策略提升检索效率
- ✅ 批处理减少API调用
- ✅ 性能监控和优化

### 4. 生产级实践
- ✅ 错误处理和重试机制
- ✅ 监控和日志系统
- ✅ 可靠性和稳定性保障

## 最佳实践总结

### 架构设计
1. **模块化设计**: 将功能分解为独立的组件
2. **状态管理**: 使用LangGraph管理复杂状态
3. **可扩展性**: 设计易于扩展的接口

### 性能优化
1. **缓存策略**: 对重复查询使用缓存
2. **批处理**: 批量处理减少开销
3. **异步执行**: 使用异步提高并发性能

### 可靠性
1. **错误处理**: 全面的异常处理
2. **重试机制**: 指数退避重试
3. **监控告警**: 实时监控系统状态

### 质量保障
1. **质量检查**: 自动评估输出质量
2. **人工审核**: 关键节点人工介入
3. **持续优化**: 基于反馈迭代改进

## 生产部署检查清单

- [ ] 配置管理（环境变量、配置文件）
- [ ] 错误处理和重试机制
- [ ] 日志和监控系统
- [ ] API限流和配额管理
- [ ] 缓存和性能优化
- [ ] 安全性（API Key管理、输入验证）
- [ ] 测试（单元测试、集成测试）
- [ ] 文档（API文档、部署文档）
- [ ] 备份和恢复策略
- [ ] 负载均衡和扩展性

## 进阶方向

1. **评估系统**: 自动评估RAG和Agent性能
2. **流式输出**: 实现流式响应提升用户体验
3. **多模态**: 支持图像、音频等多模态输入
4. **分布式部署**: 微服务架构和容器化
5. **Fine-tuning**: 针对特定领域微调模型

## 资源推荐

- [LangChain官方文档](https://python.langchain.com/)
- [LangGraph文档](https://langchain-ai.github.io/langgraph/)
- [LangChain GitHub](https://github.com/langchain-ai/langchain)
- [LangChain社区](https://github.com/langchain-ai/langchain/discussions)